# Chatbot Evaluation

In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [13]:
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [ ]:
# Creating a data points

from langsmith import Client
client = Client()

examples = [
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"}
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"}
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"}
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"}
        },
]

# Define the data points - These are test data
dataset_name = "Simple Chatbot Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_example(dataset_id=dataset.id,examples=examples)

LangSmithError: Failed to POST /datasets in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/datasets', '{"detail":"Forbidden"}')

# Define Metrics - LLM as a Judge

In [ ]:
import openai
from langsmith import wrappers
openai_client = wrappers.wrap_openai(openai.OpenAI())

eval_instructions = "You are an expert professional specialize in grading students' answer to questions."

def correctness(inputs:dict, outputs:dict, reference_outputs:dict)->bool:
    user_content=f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ]
    ).choices[0].message.content

    return response=="CORRECT"


In [ ]:
#Concision - checks if the actual answer is concise enough compared to the reference answer

def concision(outputs:dict, reference_outputs:dict)->bool:
    return int(len(outputs["response"])<2*len(reference_outputs["answer"]))

# Run Evaluation

In [ ]:
default_instructions = "Respond to the users question in a short, concise manner(one short sentence.)"

def my_app(questions:str, model:str = "gpt-4o-mini", instructions:str = default_instructions)->str:
    return openai_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": questions}
        ]
    ).choices[0].message.content

In [ ]:
# Call my_app for every datapoint

def ls_target(input:str)->dict:
    return {"response": my_app(input["question"])}

In [ ]:
# Run our evaluation

experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluator = [correctness, concision],
    experiment_prefix = "openai-4o-mini-chatbot-eval",
)